# 🏥 Maternal Health Risk Prediction
## ML Capstone Project
**Dataset:** Maternal Health Risk Data Set (UCI)  
**Objective:** Classify maternal health risk as Low / Mid / High using multiple ML models  
**Models Used:** Logistic Regression, Decision Tree, Random Forest, XGBoost, KNN  
**Key Steps:** EDA → Preprocessing → SMOTE → Modelling → Hyperparameter Tuning → Evaluation

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries imported')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Maternal Health Risk Data Set.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Class Distribution:')
print(df['RiskLevel'].value_counts())
print('\nMissing Values:')
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA)
We visualize class distribution, feature distributions by risk level, and feature correlations.

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.countplot(x='RiskLevel', data=df,
                   order=['low risk', 'mid risk', 'high risk'],
                   palette=['#2ecc71', '#f39c12', '#e74c3c'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.title('Class Distribution — Maternal Risk Level', fontsize=14, fontweight='bold')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
features = ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
colors   = {'low risk': '#2ecc71', 'mid risk': '#f39c12', 'high risk': '#e74c3c'}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(features):
    for risk_level, grp in df.groupby('RiskLevel'):
        axes[i].hist(grp[col], bins=20, alpha=0.6,
                     label=risk_level, color=colors[risk_level])
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()
plt.suptitle('Feature Distributions by Risk Level', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['RiskLevel'].value_counts()
axes[0].pie(counts.values,
            labels=counts.index,
            autopct='%1.1f%%',
            colors=['#2ecc71', '#f39c12', '#e74c3c'],
            startangle=140)
axes[0].set_title('Risk Level Distribution', fontsize=13)
risk_order = ['low risk', 'mid risk', 'high risk']
data_to_plot = [df[df['RiskLevel'] == r]['Age'].values for r in risk_order]
bp = axes[1].boxplot(data_to_plot, labels=risk_order,
                      patch_artist=True,
                      boxprops=dict(facecolor='#3498db', color='black'),
                      medianprops=dict(color='#e74c3c', linewidth=2))
axes[1].set_title('Age Distribution by Risk Level', fontsize=13)
axes[1].set_ylabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, hue='RiskLevel',
             palette={'low risk': '#2ecc71', 'mid risk': '#f39c12', 'high risk': '#e74c3c'})
plt.suptitle('Pairplot — All Features by Risk Level', y=1.02)
plt.show()

## 4. Data Preprocessing

**Steps:**
1. Remove duplicates
2. **Train-Test Split first** (prevents data leakage)
3. Outlier removal using IQR — applied only on training set
4. Label Encoding on target
5. StandardScaler on features
6. SMOTE on training set to handle class imbalance

> ⚠️ Note: Splitting before outlier removal is the correct order to avoid data leakage.

In [ ]:
# Step 1 — Remove duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Shape after dedup: {df.shape}')

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Step 2 — Encode target using consistent manual mapping
# high risk=0, low risk=1, mid risk=2 (alphabetical from LabelEncoder)
le = LabelEncoder()
df['RiskLevel_encoded'] = le.fit_transform(df['RiskLevel'])
print('Classes (alphabetical):', le.classes_)
print('Encoded as:', list(range(len(le.classes_))))

X = df[['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']]
y = df['RiskLevel_encoded']
feature_names = ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']

In [ ]:
# Step 3 — Train-Test Split FIRST (before any preprocessing on values)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)

In [ ]:
# Step 4 — Outlier removal using IQR on TRAIN SET ONLY
import pandas as pd

X_train_clean = X_train.copy()
y_train_clean = y_train.copy()

for col in feature_names:
    Q1 = X_train_clean[col].quantile(0.25)
    Q3 = X_train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (X_train_clean[col] >= lower) & (X_train_clean[col] <= upper)
    X_train_clean = X_train_clean[mask]
    y_train_clean = y_train_clean[mask]

print(f'Train shape after outlier removal: {X_train_clean.shape}')
X_train = X_train_clean
y_train = y_train_clean

In [ ]:
# Step 5 — StandardScaler (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('✅ Scaling done')

In [ ]:
# Step 6 — SMOTE on training set to handle class imbalance
from imblearn.over_sampling import SMOTE

print('Before SMOTE:', pd.Series(y_train).value_counts().to_dict())
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)
print('After SMOTE: ', pd.Series(y_train_res).value_counts().to_dict())
print('\n✅ All preprocessing complete. Using X_train_res/y_train_res for all models.')

## 5. Model Building
All models trained on SMOTE-resampled data (`X_train_res`, `y_train_res`) and evaluated on original test set.
Metrics reported: Accuracy, Classification Report (Precision/Recall/F1 per class), Confusion Matrix.

### 5.1 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_res, y_train_res)
y_pred_lr = lr_model.predict(X_test_scaled)

print('Logistic Regression Accuracy:', round(accuracy_score(y_test, y_pred_lr), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr,
    display_labels=le.classes_, cmap='Greens')
plt.title('Logistic Regression — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_res, y_train_res)
y_pred_dt = dt_model.predict(X_test_scaled)

print('Decision Tree Accuracy:', round(accuracy_score(y_test, y_pred_dt), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dt,
    display_labels=le.classes_, cmap='Blues')
plt.title('Decision Tree — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_res, y_train_res)
y_pred_rf = rf_model.predict(X_test_scaled)

print('Random Forest Accuracy:', round(accuracy_score(y_test, y_pred_rf), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf,
    display_labels=le.classes_, cmap='Oranges')
plt.title('Random Forest — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Importance
importances_rf = pd.Series(rf_model.feature_importances_, index=feature_names)
plt.figure(figsize=(7, 4))
importances_rf.sort_values().plot(kind='barh', color='darkorange')
plt.title('Random Forest — Feature Importance', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### 5.4 XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False)
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print('XGBoost Accuracy:', round(accuracy_score(y_test, y_pred_xgb), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb,
    display_labels=le.classes_, cmap='Reds')
plt.title('XGBoost — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Importance
importances_xgb = pd.Series(xgb_model.feature_importances_, index=feature_names)
plt.figure(figsize=(7, 4))
importances_xgb.sort_values().plot(kind='barh', color='crimson')
plt.title('XGBoost — Feature Importance', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### 5.5 K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_res, y_train_res)
y_pred_knn = knn_model.predict(X_test_scaled)

print('KNN Accuracy:', round(accuracy_score(y_test, y_pred_knn), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

## 6. Cross-Validation
5-Fold Cross-Validation on all models to verify generalization beyond a single train/test split.

In [ ]:
from sklearn.model_selection import cross_val_score

models_cv = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost':             XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False),
    'KNN':                 KNeighborsClassifier(n_neighbors=5)
}

print(f'{'Model':<25} {'CV Mean Acc':>12} {'CV Std':>10}')
print('-' * 50)
cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:<25} {scores.mean():>12.4f} {scores.std():>10.4f}')

In [ ]:
# Cross-Validation scores bar chart
cv_means = {name: scores.mean() for name, scores in cv_results.items()}
cv_stds  = {name: scores.std()  for name, scores in cv_results.items()}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cv_means.keys(), cv_means.values(),
              yerr=cv_stds.values(), capsize=5,
              color=['#3498db','#e67e22','#2ecc71','#e74c3c','#9b59b6'],
              alpha=0.85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('5-Fold Cross-Validation Accuracy (mean ± std)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (GridSearchCV)
GridSearchCV with 5-fold CV on Logistic Regression, Decision Tree, and Random Forest.

### 7.1 Logistic Regression Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [500, 1000]
}
lr_grid = GridSearchCV(LogisticRegression(random_state=42),
                       lr_params, cv=5, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train_res, y_train_res)
print('Best Params:', lr_grid.best_params_)
print('Best CV Accuracy:', round(lr_grid.best_score_, 4))

y_pred_lr_tuned = lr_grid.best_estimator_.predict(X_test_scaled)
print('Tuned LR Test Accuracy:', round(accuracy_score(y_test, y_pred_lr_tuned), 4))

In [ ]:
before = accuracy_score(y_test, y_pred_lr)
after  = accuracy_score(y_test, y_pred_lr_tuned)
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Before Tuning', 'After Tuning'], [before, after],
              color=['#95a5a6', '#3498db'], width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', fontweight='bold')
ax.set_title('Logistic Regression — Before vs After Tuning', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### 7.2 Decision Tree Tuning

In [ ]:
dt_params = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                       dt_params, cv=5, scoring='accuracy', n_jobs=-1)
dt_grid.fit(X_train_res, y_train_res)
print('Best Params:', dt_grid.best_params_)
print('Best CV Accuracy:', round(dt_grid.best_score_, 4))

y_pred_dt_tuned = dt_grid.best_estimator_.predict(X_test_scaled)
print('Tuned DT Test Accuracy:', round(accuracy_score(y_test, y_pred_dt_tuned), 4))

In [ ]:
before = accuracy_score(y_test, y_pred_dt)
after  = accuracy_score(y_test, y_pred_dt_tuned)
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Before Tuning', 'After Tuning'], [before, after],
              color=['#95a5a6', '#e67e22'], width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', fontweight='bold')
ax.set_title('Decision Tree — Before vs After Tuning', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### 7.3 Random Forest Tuning

In [ ]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                       rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_res, y_train_res)
print('Best Params:', rf_grid.best_params_)
print('Best CV Accuracy:', round(rf_grid.best_score_, 4))

y_pred_rf_tuned = rf_grid.best_estimator_.predict(X_test_scaled)
print('Tuned RF Test Accuracy:', round(accuracy_score(y_test, y_pred_rf_tuned), 4))

In [ ]:
before = accuracy_score(y_test, y_pred_rf)
after  = accuracy_score(y_test, y_pred_rf_tuned)
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Before Tuning', 'After Tuning'], [before, after],
              color=['#95a5a6', '#2ecc71'], width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', fontweight='bold')
ax.set_title('Random Forest — Before vs After Tuning', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### 7.4 XGBoost Tuning

In [ ]:
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0]
}
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False),
    xgb_params, cv=5, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X_train_res, y_train_res)
print('Best Params:', xgb_grid.best_params_)
print('Best CV Accuracy:', round(xgb_grid.best_score_, 4))

y_pred_xgb_tuned = xgb_grid.best_estimator_.predict(X_test_scaled)
print('Tuned XGB Test Accuracy:', round(accuracy_score(y_test, y_pred_xgb_tuned), 4))

## 8. ROC-AUC Analysis (One-vs-Rest)
ROC-AUC is more informative than accuracy for multiclass imbalanced problems. Higher AUC = better discrimination between risk classes.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Binarize labels for OvR
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes  = 3
class_names = le.classes_

models_for_roc = {
    'Logistic Regression': lr_model,
    'Decision Tree':       dt_model,
    'Random Forest':       rf_model,
    'XGBoost':             xgb_model,
    'Tuned RF':            rf_grid.best_estimator_,
    'Tuned XGB':           xgb_grid.best_estimator_
}

fig, axes = plt.subplots(1, n_classes, figsize=(18, 5))
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#1abc9c']

for cls_idx in range(n_classes):
    ax = axes[cls_idx]
    for (name, model), color in zip(models_for_roc.items(), colors):
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_test_scaled)[:, cls_idx]
        else:
            proba = model.decision_function(X_test_scaled)[:, cls_idx]
        fpr, tpr, _ = roc_curve(y_test_bin[:, cls_idx], proba)
        auc = roc_auc_score(y_test_bin[:, cls_idx], proba)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)
    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_title(f'ROC — {class_names[cls_idx]}', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=7)

plt.suptitle('ROC Curves — One-vs-Rest per Risk Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Final Model Comparison
Summary table with Accuracy, F1-Score (weighted), and ROC-AUC (macro OvR) for all models.

In [ ]:
from sklearn.metrics import f1_score

all_models = {
    'Logistic Regression':       (lr_model,              y_pred_lr),
    'Decision Tree':             (dt_model,              y_pred_dt),
    'Random Forest':             (rf_model,              y_pred_rf),
    'XGBoost':                   (xgb_model,             y_pred_xgb),
    'KNN':                       (knn_model,             y_pred_knn),
    'Tuned Logistic Regression': (lr_grid.best_estimator_, y_pred_lr_tuned),
    'Tuned Decision Tree':       (dt_grid.best_estimator_, y_pred_dt_tuned),
    'Tuned Random Forest':       (rf_grid.best_estimator_, y_pred_rf_tuned),
    'Tuned XGBoost':             (xgb_grid.best_estimator_, y_pred_xgb_tuned),
}

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
rows = []
for name, (model, y_pred) in all_models.items():
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    proba = model.predict_proba(X_test_scaled)
    auc  = roc_auc_score(y_test_bin, proba, multi_class='ovr', average='macro')
    rows.append({'Model': name, 'Accuracy': round(acc,4),
                 'F1 (weighted)': round(f1,4), 'ROC-AUC (macro)': round(auc,4)})

results_df = pd.DataFrame(rows).sort_values('ROC-AUC (macro)', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart — Accuracy vs F1 vs AUC
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(results_df))
w = 0.25
ax.bar(x - w, results_df['Accuracy'],        width=w, label='Accuracy',        color='#3498db')
ax.bar(x,     results_df['F1 (weighted)'],   width=w, label='F1 (weighted)',   color='#2ecc71')
ax.bar(x + w, results_df['ROC-AUC (macro)'], width=w, label='ROC-AUC (macro)', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison — Accuracy vs F1 vs ROC-AUC', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Conclusion

### Key Findings
- **Best Model:** XGBoost (Tuned) achieved the highest ROC-AUC (macro OvR) among all models
- **Most Important Feature:** Blood Sugar (BS) and SystolicBP consistently ranked as top predictors across RF and XGBoost
- **SMOTE Impact:** Balancing the training set improved recall for the minority 'high risk' class
- **Hyperparameter Tuning:** GridSearchCV improved performance for all tuned models over their defaults

### Clinical Relevance
- High Blood Sugar and elevated Systolic BP are strong indicators of maternal risk — consistent with medical literature
- The model shows strong ability to identify **high risk** cases (high precision + recall), which is the most critical class clinically

### Future Scope
- Deploy as a clinical decision support tool
- Add SHAP explainability for individual predictions
- Collect more data to improve mid-risk classification
- Explore deep learning models (MLP, TabNet)